In [1]:
print("Fraud Detection Project Started")

Fraud Detection Project Started


In [2]:
import pandas as pd

In [3]:
file_path = "../data/raw/PS_20174392719_1491204439457_log.csv"

df = pd.read_csv(file_path)

print("Dataset loaded successfully!")

Dataset loaded successfully!


In [4]:
df.shape

(6362620, 11)

In [5]:
df.head()

,step,type,amount,nameOrig,oldbalanceOrg,newbalanceOrig,nameDest,oldbalanceDest,newbalanceDest,isFraud,isFlaggedFraud
0,1,PAYMENT,9839.64,C1231006815,170136.0,160296.36,M1979787155,0.0,0.0,0,0
1,1,PAYMENT,1864.28,C1666544295,21249.0,19384.72,M2044282225,0.0,0.0,0,0
2,1,TRANSFER,181.00,C1305486145,181.0,0.00,C553264065,0.0,0.0,1,0
3,1,CASH_OUT,181.00,C840083671,181.0,0.00,C38997010,21182.0,0.0,1,0
4,1,PAYMENT,11668.14,C2048537720,41554.0,29885.86,M1230701703,0.0,0.0,0,0


In [6]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 6362620 entries, 0 to 6362619
Data columns (total 11 columns):
 #   Column          Dtype  
---  ------          -----  
 0   step            int64  
 1   type            object 
 2   amount          float64
 3   nameOrig        object 
 4   oldbalanceOrg   float64
 5   newbalanceOrig  float64
 6   nameDest        object 
 7   oldbalanceDest  float64
 8   newbalanceDest  float64
 9   isFraud         int64  
 10  isFlaggedFraud  int64  
dtypes: float64(5), int64(3), object(3)
memory usage: 534.0+ MB


In [7]:
df.isnull().sum()


step              0
type              0
amount            0
nameOrig          0
oldbalanceOrg     0
newbalanceOrig    0
nameDest          0
oldbalanceDest    0
newbalanceDest    0
isFraud           0
isFlaggedFraud    0
dtype: int64

In [8]:
df["isFraud"].value_counts()

isFraud
0    6354407
1       8213
Name: count, dtype: int64

In [9]:
df["type"].value_counts()

type
CASH_OUT    2237500
PAYMENT     2151495
CASH_IN     1399284
TRANSFER     532909
DEBIT         41432
Name: count, dtype: int64

In [10]:
pd.crosstab(df["type"], df["isFraud"])

isFraud,0,1
type,,
CASH_IN,1399284,0
CASH_OUT,2233384,4116
DEBIT,41432,0
PAYMENT,2151495,0
TRANSFER,528812,4097


In [11]:
df.groupby("isFraud")["amount"].describe()

,count,mean,std,min,25%,50%,75%,max
isFraud,,,,,,,,
0,6354407.0,1.781970e+05,5.962370e+05,0.01,13368.395,74684.72,208364.76,92445516.64
1,8213.0,1.467967e+06,2.404253e+06,0.00,127091.330,441423.44,1517771.48,10000000.00


In [12]:
df["balance_error_orig"] = (
    df["oldbalanceOrg"] - df["amount"] - df["newbalanceOrig"]
)

df["balance_error_orig"].describe()

count    6.362620e+06
mean    -2.010925e+05
std      6.066505e+05
min     -9.244552e+07
25%     -2.496411e+05
50%     -6.867726e+04
75%     -2.954230e+03
max      1.000000e-02
Name: balance_error_orig, dtype: float64

In [13]:
df.groupby("isFraud")["balance_error_orig"].describe()

,count,mean,std,min,25%,50%,75%,max
isFraud,,,,,,,,
0,6354407.0,-201338.558109,606928.890826,-92445516.64,-249953.43,-69049.31,-3034.305,1.000000e-02
1,8213.0,-10692.325265,265146.131130,-10000000.00,0.00,0.00,0.000,3.725290e-09


In [14]:
df["balance_error_dest"] = (
    df["oldbalanceDest"] + df["amount"] - df["newbalanceDest"]
)

df["balance_error_dest"].describe()

count    6.362620e+06
mean     5.556717e+04
std      4.415288e+05
min     -7.588573e+07
25%      0.000000e+00
50%      3.500490e+03
75%      2.935305e+04
max      1.319123e+07
Name: balance_error_dest, dtype: float64

In [15]:
df.groupby("isFraud")["balance_error_dest"].describe()

,count,mean,std,min,25%,50%,75%,max
isFraud,,,,,,,,
0,6354407.0,54692.231734,4.360026e+05,-75885725.63,0.0,3500.68,29259.805,13191233.98
1,8213.0,732509.301069,1.867748e+06,-8875516.29,0.0,2231.46,442722.010,10000000.00


In [16]:
df.groupby("isFraud")["step"].describe()

,count,mean,std,min,25%,50%,75%,max
isFraud,,,,,,,,
0,6354407.0,243.235663,142.140194,1.0,156.0,239.0,334.0,718.0
1,8213.0,368.413856,216.388690,1.0,181.0,367.0,558.0,743.0


In [17]:
df[df["isFraud"] == 1]["step"].value_counts().sort_index()

step
1      16
2       8
3       4
4      10
5       6
       ..
739    10
740     6
741    22
742    14
743     8
Name: count, Length: 741, dtype: int64

In [18]:
df["isFlaggedFraud"].value_counts()

isFlaggedFraud
0    6362604
1         16
Name: count, dtype: int64

In [19]:
fraud_rate_by_type = (
    df.groupby("type")["isFraud"]
    .mean()
    .mul(100)
    .sort_values(ascending=False)
)

fraud_rate_by_type

type
TRANSFER    0.768799
CASH_OUT    0.183955
CASH_IN     0.000000
DEBIT       0.000000
PAYMENT     0.000000
Name: isFraud, dtype: float64

In [20]:
df["amount_bin"] = pd.qcut(
    df["amount"],
    q=5,
    duplicates="drop"
)

amount_fraud_rate = (
    df.groupby("amount_bin", observed=True)["isFraud"]
    .mean()
    .mul(100)
)

amount_fraud_rate

amount_bin
(-0.001, 9866.158]          0.021689
(9866.158, 36371.35]        0.040314
(36371.35, 122563.784]      0.095008
(122563.784, 246611.22]     0.085185
(246611.22, 92445516.64]    0.403214
Name: isFraud, dtype: float64

In [21]:
df.groupby("type")["amount"].describe()

,count,mean,std,min,25%,50%,75%,max
type,,,,,,,,
CASH_IN,1399284.0,168920.242004,1.265083e+05,0.04,70510.1825,143427.710,239899.0875,1915267.90
CASH_OUT,2237500.0,176273.964346,1.753297e+05,0.00,72669.6500,147072.185,246539.4775,10000000.00
DEBIT,41432.0,5483.665314,1.331854e+04,0.55,1500.1800,3048.990,5479.1750,569077.51
PAYMENT,2151495.0,13057.604660,1.255645e+04,0.02,4383.8200,9482.190,17561.2200,238637.98
TRANSFER,532909.0,910647.009645,1.879574e+06,2.60,215905.3500,486308.390,974958.0000,92445516.64


In [22]:
# Fraud rate by transaction type and amount range

df["amount_bin_type"] = (
    df.groupby("type")["amount"]
    .transform(
        lambda x: pd.qcut(
            x,
            q=5,
            labels=False,
            duplicates="drop"
        )
    )
)

fraud_rate_type_amount = (
    df.groupby(["type", "amount_bin_type"], observed=True)["isFraud"]
    .mean()
    .mul(100)
)

fraud_rate_type_amount

type      amount_bin_type
CASH_IN   0                  0.000000
          1                  0.000000
          2                  0.000000
          3                  0.000000
          4                  0.000000
CASH_OUT  0                  0.130726
          1                  0.087151
          2                  0.078436
          3                  0.073743
          4                  0.549721
DEBIT     0                  0.000000
          1                  0.000000
          2                  0.000000
          3                  0.000000
          4                  0.000000
PAYMENT   0                  0.000000
          1                  0.000000
          2                  0.000000
          3                  0.000000
          4                  0.000000
TRANSFER  0                  1.163423
          1                  0.603291
          2                  0.448485
          3                  0.451296
          4                  1.177497
Name: isFraud, dtype: fl

In [23]:
# Feature engineering

df["orig_balance_change"] = df["oldbalanceOrg"] - df["newbalanceOrig"]

df["dest_balance_change"] = df["newbalanceDest"] - df["oldbalanceDest"]

df["amount_to_orig_balance"] = (
    df["amount"] / (df["oldbalanceOrg"] + 1)
)

df["amount_to_dest_balance"] = (
    df["amount"] / (df["oldbalanceDest"] + 1)
)

df[[
    "amount",
    "oldbalanceOrg",
    "newbalanceOrig",
    "oldbalanceDest",
    "newbalanceDest",
    "balance_error_orig",
    "balance_error_dest",
    "orig_balance_change",
    "dest_balance_change",
    "amount_to_orig_balance",
    "amount_to_dest_balance"
]].head()

,amount,oldbalanceOrg,newbalanceOrig,oldbalanceDest,newbalanceDest,balance_error_orig,balance_error_dest,orig_balance_change,dest_balance_change,amount_to_orig_balance,amount_to_dest_balance
0,9839.64,170136.0,160296.36,0.0,0.0,0.0,9839.64,9839.64,0.0,0.057834,9839.640000
1,1864.28,21249.0,19384.72,0.0,0.0,0.0,1864.28,1864.28,0.0,0.087731,1864.280000
2,181.00,181.0,0.00,0.0,0.0,0.0,181.00,181.00,0.0,0.994505,181.000000
3,181.00,181.0,0.00,21182.0,0.0,0.0,21363.00,181.00,-21182.0,0.994505,0.008545
4,11668.14,41554.0,29885.86,0.0,0.0,0.0,11668.14,11668.14,0.0,0.280788,11668.140000


In [24]:


# Select features for machine learning

features = [
    "step",
    "type",
    "amount",
    "oldbalanceOrg",
    "newbalanceOrig",
    "oldbalanceDest",
    "newbalanceDest",
    "isFlaggedFraud",
    "balance_error_orig",
    "balance_error_dest",
    "orig_balance_change",
    "dest_balance_change",
    "amount_to_orig_balance",
    "amount_to_dest_balance"
]

X = df[features]
y = df["isFraud"]

print("Feature matrix shape:", X.shape)
print("Target shape:", y.shape)
print("\nFeatures:")
print(X.columns.tolist())

Feature matrix shape: (6362620, 14)
Target shape: (6362620,)

Features:
['step', 'type', 'amount', 'oldbalanceOrg', 'newbalanceOrig', 'oldbalanceDest', 'newbalanceDest', 'isFlaggedFraud', 'balance_error_orig', 'balance_error_dest', 'orig_balance_change', 'dest_balance_change', 'amount_to_orig_balance', 'amount_to_dest_balance']


In [25]:
# One-hot encode transaction type

X = pd.get_dummies(
    X,
    columns=["type"],
    drop_first=True,
    dtype=int
)

print("Encoded feature matrix shape:", X.shape)
print("\nEncoded features:")
print(X.columns.tolist())

Encoded feature matrix shape: (6362620, 17)

Encoded features:
['step', 'amount', 'oldbalanceOrg', 'newbalanceOrig', 'oldbalanceDest', 'newbalanceDest', 'isFlaggedFraud', 'balance_error_orig', 'balance_error_dest', 'orig_balance_change', 'dest_balance_change', 'amount_to_orig_balance', 'amount_to_dest_balance', 'type_CASH_OUT', 'type_DEBIT', 'type_PAYMENT', 'type_TRANSFER']


In [26]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.20,
    random_state=42,
    stratify=y
)

print("Training set:", X_train.shape)
print("Testing set:", X_test.shape)

print("\nTraining fraud rate:", y_train.mean() * 100, "%")
print("Testing fraud rate:", y_test.mean() * 100, "%")

Training set: (5090096, 17)
Testing set: (1272524, 17)

Training fraud rate: 0.12907418642005966 %
Testing fraud rate: 0.12911347840983745 %


In [27]:
# Check class distribution in the training set

print("Training class distribution:")
print(y_train.value_counts())

print("\nTraining class percentages:")
print((y_train.value_counts(normalize=True) * 100))

Training class distribution:
isFraud
0    5083526
1       6570
Name: count, dtype: int64

Training class percentages:
isFraud
0    99.870926
1     0.129074
Name: proportion, dtype: float64


In [28]:
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression

baseline_model = Pipeline([
    ("scaler", StandardScaler()),
    (
        "classifier",
        LogisticRegression(
            class_weight="balanced",
            max_iter=1000,
            random_state=42
        )
    )
])

print("Training baseline Logistic Regression model...")

baseline_model.fit(X_train, y_train)

print("Baseline model training completed!")

Training baseline Logistic Regression model...
Baseline model training completed!


In [29]:
from sklearn.metrics import (
    classification_report,
    confusion_matrix,
    roc_auc_score,
    average_precision_score
)

# Make predictions on the unseen test set
y_pred_baseline = baseline_model.predict(X_test)
y_prob_baseline = baseline_model.predict_proba(X_test)[:, 1]

# Confusion matrix
print("Confusion Matrix:")
print(confusion_matrix(y_test, y_pred_baseline))

# Classification report
print("\nClassification Report:")
print(classification_report(y_test, y_pred_baseline, digits=4))

# ROC-AUC
roc_auc_baseline = roc_auc_score(y_test, y_prob_baseline)

# PR-AUC / Average Precision
pr_auc_baseline = average_precision_score(y_test, y_prob_baseline)

print(f"ROC-AUC: {roc_auc_baseline:.4f}")
print(f"PR-AUC: {pr_auc_baseline:.4f}")

Confusion Matrix:
[[1224735   46146]
 [     31    1612]]

Classification Report:
              precision    recall  f1-score   support

           0     1.0000    0.9637    0.9815   1270881
           1     0.0338    0.9811    0.0653      1643

    accuracy                         0.9637   1272524
   macro avg     0.5169    0.9724    0.5234   1272524
weighted avg     0.9987    0.9637    0.9803   1272524

ROC-AUC: 0.9962
PR-AUC: 0.6196
